# 🥉 Bronze Layer: S3 Landing Zone & Data Lakehouse Ingestion
**Purpose:** Fetch new sentiment data from Alpha Vantage, save the raw JSON payload to our AWS S3 Data Lake (Landing Zone), and then use PySpark to read that file into our Databricks Bronze table.

In [0]:
# Boto3 is the official AWS SDK for Python
%pip install boto3 
dbutils.library.restartPython()

In [0]:
import pandas as pd
import requests
import json
import boto3
from datetime import datetime, timedelta
from pyspark.sql import functions as F

In [0]:
with open("secrets.json", "r") as file:
        secrets = json.load(file)
        API_KEY = secrets.get("alpha_vantage")
        AWS_ACCESS_KEY = secrets.get("aws_access_key")
        AWS_SECRET_KEY = secrets.get("aws_secret_key")

In [0]:
# 1. Configuration
CATALOG = "portfolio"
SCHEMA = "market_data"
BRONZE_NEWS_TABLE = f"{CATALOG}.{SCHEMA}.bronze_news_sentiment"
TICKER = "AAPL"
BUCKET_NAME = "portfolio-market-data-raw-pedro-franca"

# Create Catalog, Database if it doesn't exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG}.{SCHEMA}")

In [0]:
# 2. Fetch Data
## From Alpha Vantage (last 24 hours)
time_24h_ago = datetime.now() - timedelta(hours=24)
formatted_time_from = time_24h_ago.strftime("%Y%m%dT%H%M")
url = (
    f"https://www.alphavantage.co/query?"
    f"function=NEWS_SENTIMENT&"
    f"tickers={TICKER}&"
    f"time_from={formatted_time_from}&"
    f"sort=LATEST&"
    f"limit=100&"
    f"apikey={API_KEY}"
)

response = requests.get(url)
data = response.json()

# The actual news articles are inside the 'feed' key
news_feed = data.get("feed", [])

In [0]:
news_feed

In [0]:
# If no news is found, exit the notebook gracefully
if not news_feed:
    print("No news found in the last 24 hours. Exiting notebook gracefully.")
    dbutils.notebook.exit("Success: No new data to process.")

In [0]:
# 3. Load to Cloud: Save Raw JSON to AWS S3
# Create a unique file name using today's timestamp
file_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
s3_file_key = f"alphavantage_drops/{TICKER}_news_{file_timestamp}.json"
s3_path = f"s3://{BUCKET_NAME}/{s3_file_key}"

# Connect to S3 using boto3
s3_client = boto3.client(
    's3',
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY
)

# Upload the raw JSON dictionary to S3
s3_client.put_object(
    Bucket=BUCKET_NAME,
    Key=s3_file_key,
    Body=json.dumps(news_feed)
)
print(f"Successfully saved raw data to {s3_path}")

In [0]:
# 4. Load to Lakehouse: Create Bronze Table
# Instead of making Spark re-download from S3 (which Serverless blocks with access keys), 
# we create the DataFrame directly from our local Python dictionary.
df_bronze_news = spark.createDataFrame(news_feed)

# Add Data Governance Metadata (Lineage)
df_bronze_news = df_bronze_news.withColumn("ticker_symbol", F.lit(TICKER)) \
                               .withColumn("ingestion_timestamp", F.current_timestamp()) \
                               .withColumn("source_system", F.lit("alphavantage_api")) \
                               .withColumn("s3_source_file", F.lit(s3_path)) # Keeps our S3 traceability intact!

# Write to Bronze (Append Mode)
df_bronze_news.write.format("delta").mode("append").saveAsTable(BRONZE_NEWS_TABLE)

display(spark.read.table(BRONZE_NEWS_TABLE).orderBy(F.col("ingestion_timestamp").desc()).limit(5))